# Phân Loại Thể Loại Âm Nhạc bằng Machine Learning
**Môn:** Xử Lý Tín Hiệu Số | **Nhóm:** ABNQ

## Mục tiêu
- Phân loại **10 thể loại âm nhạc** từ GTZAN dataset
- Trích xuất đặc trưng âm thanh: **MFCC, Chroma, Spectral Contrast**
- So sánh 2 giải thuật: **SVM** và **Random Forest**
- Đánh giá qua confusion matrix và classification report

## Dataset
| Phần | Đường dẫn | Số lượng |
|------|-----------|----------|
| Training | `train-data/original-genres/` | ~900 files |
| Testing | `test-data/` | ~100 files |
| Genres | blues, classical, country, disco, hiphop, jazz, metal, pop, reggae, rock | 10 |

## Pipeline
```
Audio (.wav) → Feature Extraction → StandardScaler → SVM / RF → Prediction
```

## 1. Import Thư Viện

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
import warnings
warnings.filterwarnings('ignore')

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, confusion_matrix,
                              classification_report, ConfusionMatrixDisplay)
from sklearn.pipeline import Pipeline

# Kiểm tra phiên bản
import sklearn
print(f'NumPy   : {np.__version__}')
print(f'Librosa : {librosa.__version__}')
print(f'sklearn : {sklearn.__version__}')
print('Tất cả thư viện OK!')

## 2. Cấu Hình Dataset

In [ ]:
# Đường dẫn dataset
TRAIN_DIR = 'train-data/original-genres'
TEST_DIR  = 'test-data'

# 10 thể loại nhạc
GENRES = sorted(['blues', 'classical', 'country', 'disco', 'hiphop',
                 'jazz', 'metal', 'pop', 'reggae', 'rock'])

# Tham số feature extraction
SAMPLE_RATE = 22050   # Hz (librosa default)
DURATION    = 30      # giây (GTZAN chuẩn)
N_MFCC      = 20      # Số hệ số MFCC
N_CHROMA    = 12      # Số bins Chroma
N_CONTRAST  = 7       # Số bands Spectral Contrast

# Kiểm tra dataset tồn tại
print('Kiểm tra dataset...')
for split, path in [('Train', TRAIN_DIR), ('Test', TEST_DIR)]:
    if os.path.exists(path):
        genres_found = [d for d in os.listdir(path)
                        if os.path.isdir(os.path.join(path, d))]
        total_files = sum(
            len([f for f in os.listdir(os.path.join(path, g)) if f.endswith('.wav')])
            for g in genres_found if os.path.isdir(os.path.join(path, g))
        )
        print(f'  {split}: {path} — {len(genres_found)} genres, ~{total_files} files')
    else:
        print(f'  {split}: {path} — KHÔNG TÌM THẤY!')

print(f'\n10 thể loại: {GENRES}')

## 3. Trích Xuất Đặc Trưng Âm Thanh

### Các features được dùng:

| Feature | Số chiều | Mô tả |
|---------|---------|-------|
| MFCC (mean + std) | 40 | Đặc trưng cepstral, mô phỏng thính giác người |
| Chroma STFT (mean + std) | 24 | Phân bố năng lượng theo 12 note nhạc |
| Spectral Contrast (mean + std) | 14 | Tỷ lệ năng lượng đỉnh/thung trong từng sub-band |
| ZCR (mean + std) | 2 | Tốc độ đổi dấu — liên quan đến nhịp điệu |
| RMS Energy (mean + std) | 2 | Mức năng lượng bình quân |
| **Tổng** | **82** | |

In [ ]:
def extract_features(file_path, sr=SAMPLE_RATE, duration=DURATION):
    """
    Trích xuất feature vector từ file audio.
    
    Tham số:
        file_path : đường dẫn file .wav
        sr        : tần số lấy mẫu (Hz)
        duration  : thời lượng tối đa (giây)
    
    Trả về:
        features : numpy array shape=(82,)
    """
    try:
        # Load audio (tự resample về sr, lấy tối đa duration giây)
        y, sr = librosa.load(file_path, sr=sr, duration=duration, mono=True)
        
        features = []
        
        # 1. MFCC — Mel-Frequency Cepstral Coefficients
        # Mô phỏng cách tai người cảm nhận tần số (thang Mel phi tuyến)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
        features.extend(np.mean(mfcc, axis=1))   # 20 features
        features.extend(np.std(mfcc, axis=1))    # 20 features
        
        # 2. Chroma STFT — Phân bố năng lượng theo 12 note nhạc (C, C#, D...)
        chroma = librosa.feature.chroma_stft(y=y, sr=sr, n_chroma=N_CHROMA)
        features.extend(np.mean(chroma, axis=1))  # 12 features
        features.extend(np.std(chroma, axis=1))   # 12 features
        
        # 3. Spectral Contrast — Độ tương phản phổ giữa đỉnh và thung
        contrast = librosa.feature.spectral_contrast(y=y, sr=sr,
                                                      n_bands=N_CONTRAST - 1)
        features.extend(np.mean(contrast, axis=1))  # 7 features
        features.extend(np.std(contrast, axis=1))   # 7 features
        
        # 4. Zero Crossing Rate — Tốc độ đổi dấu (liên quan nhịp điệu)
        zcr = librosa.feature.zero_crossing_rate(y)
        features.append(np.mean(zcr))   # 1 feature
        features.append(np.std(zcr))    # 1 feature
        
        # 5. RMS Energy — Mức năng lượng bình quân
        rms = librosa.feature.rms(y=y)
        features.append(np.mean(rms))   # 1 feature
        features.append(np.std(rms))    # 1 feature
        
        return np.array(features, dtype=np.float32)
    
    except Exception as e:
        print(f'Lỗi: {file_path} — {e}')
        return None


# Test với 1 file
test_files = []
for genre in GENRES:
    genre_path = os.path.join(TRAIN_DIR, genre)
    if os.path.exists(genre_path):
        files = [f for f in os.listdir(genre_path) if f.endswith('.wav')]
        if files:
            test_files.append(os.path.join(genre_path, files[0]))
            break

if test_files:
    sample = extract_features(test_files[0])
    print(f'Test feature extraction:')
    print(f'  File   : {test_files[0]}')
    print(f'  Shape  : {sample.shape}')
    print(f'  Range  : [{sample.min():.3f}, {sample.max():.3f}]')
    print(f'  82 features = 40 MFCC + 24 Chroma + 14 Contrast + 2 ZCR + 2 RMS')

## 4. Trích Xuất Features Cho Toàn Bộ Dataset

In [ ]:
def load_dataset(data_dir, genres=GENRES, verbose=True):
    """
    Load và extract features toàn bộ dataset.
    
    Trả về:
        X      : feature matrix, shape=(n_samples, 82)
        y      : labels (genre names), shape=(n_samples,)
        errors : danh sách file bị lỗi
    """
    X, y, errors = [], [], []
    
    for genre in genres:
        genre_path = os.path.join(data_dir, genre)
        if not os.path.exists(genre_path):
            if verbose:
                print(f'  Bỏ qua (không tìm thấy): {genre_path}')
            continue
        
        files = sorted([f for f in os.listdir(genre_path) if f.endswith('.wav')])
        count_ok = 0
        
        for fname in files:
            fpath = os.path.join(genre_path, fname)
            features = extract_features(fpath)
            
            if features is not None:
                X.append(features)
                y.append(genre)
                count_ok += 1
            else:
                errors.append(fpath)
        
        if verbose:
            print(f'  {genre:12s}: {count_ok}/{len(files)} files OK')
    
    return np.array(X), np.array(y), errors


print('=== TRAINING SET ===')
X_train, y_train, train_errors = load_dataset(TRAIN_DIR)
print(f'Training: X={X_train.shape}, y={y_train.shape}')

print()
print('=== TEST SET ===')
X_test, y_test, test_errors = load_dataset(TEST_DIR)
print(f'Test: X={X_test.shape}, y={y_test.shape}')

if train_errors or test_errors:
    print(f'\nLỗi: {len(train_errors)} train files, {len(test_errors)} test files')

## 5. Phân Tích Dữ Liệu (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of genres
from collections import Counter
train_counts = Counter(y_train)
test_counts  = Counter(y_test)

genres_sorted = sorted(train_counts.keys())
train_vals = [train_counts[g] for g in genres_sorted]
test_vals  = [test_counts.get(g, 0) for g in genres_sorted]

x = np.arange(len(genres_sorted))
width = 0.35
axes[0].bar(x - width/2, train_vals, width, label='Train', color='steelblue', alpha=0.8)
axes[0].bar(x + width/2, test_vals,  width, label='Test',  color='orange',    alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(genres_sorted, rotation=45, ha='right')
axes[0].set_ylabel('Số lượng file')
axes[0].set_title('Phân phối Genre trong Dataset')
axes[0].legend()
axes[0].grid(True, axis='y', alpha=0.3)

# Feature distribution visualization (MFCC 1)
for g in genres_sorted:
    mask = y_train == g
    axes[1].hist(X_train[mask, 0], bins=20, alpha=0.5, label=g, density=True)
axes[1].set_xlabel('MFCC[0] — Mean của hệ số MFCC đầu tiên')
axes[1].set_ylabel('Mật độ xác suất')
axes[1].set_title('Phân phối MFCC[0] theo Genre (Train set)')
axes[1].legend(fontsize=7, ncol=2)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('data_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Hình 5: Phân phối dữ liệu và đặc trưng MFCC theo genre')

## 6. Trực Quan Hóa Mel Spectrogram
Mel Spectrogram là biểu diễn trực quan thường dùng trong phân tích âm thanh.

In [ ]:
# Vẽ Mel Spectrogram cho 4 genres
show_genres = ['blues', 'classical', 'rock', 'jazz']
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Mel Spectrogram — So Sánh Giữa Các Thể Loại Nhạc', fontsize=13, fontweight='bold')

for ax, genre in zip(axes.flatten(), show_genres):
    genre_path = os.path.join(TRAIN_DIR, genre)
    if os.path.exists(genre_path):
        files = [f for f in os.listdir(genre_path) if f.endswith('.wav')]
        if files:
            fpath = os.path.join(genre_path, files[0])
            y_audio, sr = librosa.load(fpath, sr=22050, duration=10)
            S = librosa.feature.melspectrogram(y=y_audio, sr=sr, n_mels=128)
            S_db = librosa.power_to_db(S, ref=np.max)
            img = librosa.display.specshow(S_db, sr=sr, x_axis='time',
                                           y_axis='mel', ax=ax, cmap='viridis')
            ax.set_title(f'{genre.upper()}', fontsize=12, fontweight='bold')
            ax.set_xlabel('Thời gian (giây)')
            ax.set_ylabel('Tần số (Mel)')
            plt.colorbar(img, ax=ax, format='%+2.0f dB')

plt.tight_layout()
plt.savefig('mel_spectrograms.png', dpi=150, bbox_inches='tight')
plt.show()
print('Hình 6: Mel Spectrogram của 4 thể loại nhạc')
print('Nhận xét: Mỗi genre có pattern Mel Spectrogram đặc trưng.')

## 7. Chuẩn Hóa Dữ Liệu (StandardScaler)

In [ ]:
# Mã hóa nhãn (genre string → integer)
le = LabelEncoder()
le.fit(GENRES)
y_train_enc = le.transform(y_train)
y_test_enc  = le.transform(y_test)

# StandardScaler: chuẩn hóa về mean=0, std=1
# Rất quan trọng với SVM vì SVM nhạy cảm với scale của features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit trên train, transform train
X_test_scaled  = scaler.transform(X_test)        # chỉ transform test (KHÔNG fit lại)

print('Chuẩn hóa dữ liệu (StandardScaler):')
print(f'  X_train trước: mean={X_train.mean():.3f}, std={X_train.std():.3f}')
print(f'  X_train sau  : mean={X_train_scaled.mean():.6f}, std={X_train_scaled.std():.6f}')
print()
print(f'  X_test shape : {X_test_scaled.shape}')
print()
print(f'Classes: {list(le.classes_)}')

## 8. Huấn Luyện Model — SVM

### Cơ sở lý thuyết SVM
SVM tìm **siêu phẳng phân tách** tối ưu với **margin lớn nhất**.

Với Kernel RBF (Radial Basis Function):
$$K(x, x') = \exp\left(-\gamma \|x - x'\|^2\right)$$

- `C`: hệ số phạt — C lớn → ít lỗi training nhưng có thể overfit
- `gamma`: độ rộng kernel — gamma nhỏ → smooth boundary

In [ ]:
print('Huấn luyện SVM (RBF kernel)...')

svm_model = SVC(
    kernel='rbf',
    C=10,
    gamma='scale',
    probability=True,      # Cần để lấy confidence score
    random_state=42,
    class_weight='balanced'  # Xử lý class imbalance
)

svm_model.fit(X_train_scaled, y_train_enc)

# Đánh giá
y_pred_svm  = svm_model.predict(X_test_scaled)
acc_svm     = accuracy_score(y_test_enc, y_pred_svm)
acc_train_svm = accuracy_score(y_train_enc, svm_model.predict(X_train_scaled))

print(f'SVM Training accuracy : {acc_train_svm:.4f} ({acc_train_svm*100:.1f}%)')
print(f'SVM Test accuracy     : {acc_svm:.4f} ({acc_svm*100:.1f}%)')
print(f'Support Vectors       : {svm_model.n_support_.sum()} vectors')

## 9. Huấn Luyện Model — Random Forest

### Cơ sở lý thuyết Random Forest
Random Forest là **ensemble** của nhiều Decision Trees:
- Mỗi tree được train trên **bootstrap sample** (lấy mẫu có hoàn lại)
- Mỗi split chỉ xét **subset ngẫu nhiên** của features → giảm tương quan giữa trees
- Kết quả: **voting đa số** → giảm variance, tránh overfitting

In [ ]:
print('Huấn luyện Random Forest...')

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,        # Cho phép cây sâu đủ
    min_samples_split=2,
    random_state=42,
    n_jobs=-1,             # Dùng tất cả CPU cores
    class_weight='balanced'
)

rf_model.fit(X_train_scaled, y_train_enc)

# Đánh giá
y_pred_rf  = rf_model.predict(X_test_scaled)
acc_rf     = accuracy_score(y_test_enc, y_pred_rf)
acc_train_rf = accuracy_score(y_train_enc, rf_model.predict(X_train_scaled))

print(f'RF Training accuracy : {acc_train_rf:.4f} ({acc_train_rf*100:.1f}%)')
print(f'RF Test accuracy     : {acc_rf:.4f} ({acc_rf*100:.1f}%)')
print(f'Số cây               : {rf_model.n_estimators}')

## 10. So Sánh Kết Quả Hai Mô Hình

In [ ]:
print('=' * 50)
print('   SO SÁNH KẾT QUẢ')
print('=' * 50)
print(f'{"Model":<20} {"Train Acc":>10} {"Test Acc":>10}')
print('-' * 45)
print(f'{"SVM (RBF)":<20} {acc_train_svm:>10.1%} {acc_svm:>10.1%}')
print(f'{"Random Forest":<20} {acc_train_rf:>10.1%} {acc_rf:>10.1%}')

best_model = svm_model if acc_svm >= acc_rf else rf_model
best_name  = 'SVM' if acc_svm >= acc_rf else 'Random Forest'
best_acc   = max(acc_svm, acc_rf)
y_pred_best = y_pred_svm if acc_svm >= acc_rf else y_pred_rf

print(f'\nMô hình tốt hơn: {best_name} ({best_acc:.1%})')

# Bar chart so sánh
fig, ax = plt.subplots(figsize=(8, 5))
models  = ['SVM\n(RBF)', 'Random\nForest']
train_accs = [acc_train_svm, acc_train_rf]
test_accs  = [acc_svm, acc_rf]

x = np.arange(len(models))
w = 0.35
ax.bar(x - w/2, [a*100 for a in train_accs], w, label='Train', color='steelblue', alpha=0.8)
ax.bar(x + w/2, [a*100 for a in test_accs],  w, label='Test',  color='orange',    alpha=0.8)

for i, (tr, te) in enumerate(zip(train_accs, test_accs)):
    ax.text(i - w/2, tr*100 + 0.5, f'{tr:.1%}', ha='center', fontsize=10)
    ax.text(i + w/2, te*100 + 0.5, f'{te:.1%}', ha='center', fontsize=10)

ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=11)
ax.set_ylabel('Accuracy (%)')
ax.set_title('So Sánh Độ Chính Xác: SVM vs Random Forest')
ax.set_ylim([0, 110])
ax.legend(fontsize=11)
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Confusion Matrix — Phân Loại Thể Loại Âm Nhạc', fontsize=13, fontweight='bold')

for ax, (name, y_pred) in zip(axes, [('SVM', y_pred_svm), ('Random Forest', y_pred_rf)]):
    cm = confusion_matrix(y_test_enc, y_pred)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=GENRES, yticklabels=GENRES,
                ax=ax, linewidths=0.5)
    ax.set_title(f'{name}  (Accuracy: {accuracy_score(y_test_enc, y_pred):.1%})',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted Label', fontsize=11)
    ax.set_ylabel('True Label', fontsize=11)
    ax.tick_params(axis='x', rotation=45)
    ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Hình 7: Confusion matrix — đường chéo = dự đoán đúng')
print('Nhận xét: Ô nào sáng ngoài đường chéo = nhầm lẫn nhiều nhất')

## 12. Classification Report Chi Tiết

In [ ]:
print('=== SVM — Classification Report ===')
print(classification_report(y_test_enc, y_pred_svm,
                             target_names=GENRES, digits=3))

print()
print('=== Random Forest — Classification Report ===')
print(classification_report(y_test_enc, y_pred_rf,
                             target_names=GENRES, digits=3))

## 13. Feature Importance (Random Forest)

In [ ]:
# Tên các features
feature_names = []
feature_names += [f'MFCC_mean_{i}' for i in range(N_MFCC)]
feature_names += [f'MFCC_std_{i}'  for i in range(N_MFCC)]
feature_names += [f'Chroma_mean_{i}' for i in range(N_CHROMA)]
feature_names += [f'Chroma_std_{i}'  for i in range(N_CHROMA)]
feature_names += [f'Contrast_mean_{i}' for i in range(N_CONTRAST)]
feature_names += [f'Contrast_std_{i}'  for i in range(N_CONTRAST)]
feature_names += ['ZCR_mean', 'ZCR_std', 'RMS_mean', 'RMS_std']

# Top 20 features quan trọng nhất
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1][:20]

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(range(20), importances[indices[::-1]], color='steelblue', alpha=0.8)
ax.set_yticks(range(20))
ax.set_yticklabels([feature_names[i] for i in indices[::-1]], fontsize=9)
ax.set_xlabel('Feature Importance (Gini)', fontsize=11)
ax.set_title('Top 20 Features Quan Trọng Nhất (Random Forest)', fontsize=12)
ax.grid(True, axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Hình 8: Top 20 features quan trọng nhất')

## 14. Dự Đoán Thể Loại File Âm Nhạc Mới

In [ ]:
def predict_genre(file_path, model, scaler, label_encoder):
    """
    Dự đoán thể loại của 1 file audio.
    
    Trả về:
        predicted_genre : tên thể loại dự đoán
        confidence      : độ tin cậy (%)
        top3            : top 3 dự đoán và confidence
    """
    features = extract_features(file_path)
    if features is None:
        return None, None, None
    
    features_scaled = scaler.transform(features.reshape(1, -1))
    
    # Dự đoán
    pred_idx = model.predict(features_scaled)[0]
    proba    = model.predict_proba(features_scaled)[0]
    
    predicted_genre = label_encoder.inverse_transform([pred_idx])[0]
    confidence      = proba[pred_idx] * 100
    
    # Top 3
    top3_idx = np.argsort(proba)[::-1][:3]
    top3 = [(label_encoder.inverse_transform([i])[0], proba[i]*100) for i in top3_idx]
    
    return predicted_genre, confidence, top3


# ============================================
# ĐỔI ĐƯỜNG DẪN FILE ÂM NHẠC CẦN DỰ ĐOÁN
# ============================================
PREDICT_FILE = 'test-data/blues/blues.00081.wav'  # Ví dụ

if os.path.exists(PREDICT_FILE):
    best_pred, best_conf, top3 = predict_genre(PREDICT_FILE, best_model, scaler, le)
    
    print(f'File     : {PREDICT_FILE}')
    print(f'Dự đoán  : {best_pred.upper()} (Độ tin cậy: {best_conf:.1f}%)')
    print(f'Mô hình  : {best_name}')
    print()
    print('Top 3 dự đoán:')
    for i, (genre, conf) in enumerate(top3, 1):
        bar = '█' * int(conf / 5)
        print(f'  {i}. {genre:12s}: {conf:5.1f}%  {bar}')
    
    # Visualize confidence
    fig, ax = plt.subplots(figsize=(10, 4))
    features_scaled = scaler.transform(extract_features(PREDICT_FILE).reshape(1, -1))
    proba = best_model.predict_proba(features_scaled)[0]
    genre_proba = list(zip(le.classes_, proba * 100))
    genre_proba.sort(key=lambda x: x[1], reverse=True)
    
    genres_display = [g for g, _ in genre_proba]
    proba_display  = [p for _, p in genre_proba]
    colors_bar = ['gold' if g == best_pred else 'steelblue' for g in genres_display]
    
    ax.barh(genres_display, proba_display, color=colors_bar, alpha=0.85)
    ax.set_xlabel('Confidence (%)')
    ax.set_title(f'Kết Quả Phân Loại: {PREDICT_FILE.split("/")[-1]}')
    ax.axvline(50, color='red', linestyle='--', alpha=0.5, label='50%')
    ax.legend()
    ax.grid(True, axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig('prediction_result.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print(f'File không tồn tại: {PREDICT_FILE}')
    print('Đổi PREDICT_FILE thành đường dẫn đúng.')

## 15. Tóm Tắt Kết Quả

In [ ]:
print('=' * 60)
print('   TÓM TẮT KẾT QUẢ PHÂN LOẠI THỂ LOẠI ÂM NHẠC')
print('=' * 60)
print(f'Dataset train  : {TRAIN_DIR}')
print(f'Dataset test   : {TEST_DIR}')
print(f'Số genre       : {len(GENRES)}')
print(f'Genres         : {GENRES}')
print(f'Training set   : {len(X_train)} files')
print(f'Test set       : {len(X_test)} files')
print()
print(f'Feature vector : 82 features')
print(f'  MFCC         : 40 (20 mean + 20 std)')
print(f'  Chroma STFT  : 24 (12 mean + 12 std)')
print(f'  Spec Contrast: 14 ( 7 mean +  7 std)')
print(f'  ZCR          :  2 (mean + std)')
print(f'  RMS Energy   :  2 (mean + std)')
print()
print(f'SVM (RBF, C=10)    Test Accuracy: {acc_svm:.1%}')
print(f'Random Forest      Test Accuracy: {acc_rf:.1%}')
print(f'Mô hình tốt hơn  : {best_name} ({best_acc:.1%})')
print()
print('Các file đầu ra:')
print('  data_distribution.png   — Phân phối dataset')
print('  mel_spectrograms.png    — Mel Spectrogram 4 genres')
print('  model_comparison.png    — So sánh accuracy')
print('  confusion_matrix.png    — Confusion matrix')
print('  feature_importance.png  — Feature importance (RF)')
print('  prediction_result.png   — Kết quả dự đoán')